<a href="https://colab.research.google.com/github/pecorebuild-hub/colab/blob/main/TripoSR_gradio_jupyter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

```markdown
### Шаг для авторизации в Hugging Face (Необязательно, если загрузка уже работает без токена)

Для предотвращения ошибок лимитов скачивания (Rate Limit), вы можете добавить токен Hugging Face в секреты Colab (иконка ключа слева) под именем `HF_TOKEN` и запустить ячейку ниже:
```

In [10]:
import os
from google.colab import userdata

try:
    # Пробуем получить токен из секретов Colab
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
        print("✅ Токен HF_TOKEN успешно применен из секретов Colab!")
    else:
        print("ℹ️ Секрет HF_TOKEN пуст или не задан. Скачивание продолжится без авторизации.")
except Exception as e:
    print("ℹ️ Токен HF_TOKEN не найден в секретах. Если возникнут ошибки скачивания, добавьте ключ HF_TOKEN в панель Secrets (слева).")

ℹ️ Токен HF_TOKEN не найден в секретах. Если возникнут ошибки скачивания, добавьте ключ HF_TOKEN в панель Secrets (слева).


In [ ]:
# 1. Полная очистка рабочей директории
%cd /content
!rm -rf /content/TripoSR-hf

# 2. Клонирование репозитория
!git clone -b dev https://github.com/camenduru/TripoSR-hf
%cd /content/TripoSR-hf

# 3. Удаление конфликтующих CUDA ONNX библиотек и чистая установка на CPU
!pip uninstall -y onnxruntime onnxruntime-gpu
!pip install -q trimesh omegaconf einops rembg onnxruntime gradio PyMCubes

# 4. Патчим isosurface.py с полной гарантией совместимости
import os

isosurface_path = "tsr/models/isosurface.py"
if os.path.exists(isosurface_path):
    with open(isosurface_path, "r") as f:
        lines = f.readlines()

    new_lines = []
    skip_original_device_lines = False
    for line in lines:
        # Заменяем импорт
        if "from torchmcubes import marching_cubes" in line:
            new_lines.append("import mcubes\n")
            new_lines.append("marching_cubes = mcubes.marching_cubes\n")
            continue

        # Перехватываем вызов mc_func внутри метода forward
        if "v_pos, t_pos_idx = self.mc_func(level.detach(), 0.0)" in line:
            indent = line.__len__() - line.lstrip().__len__()
            spaces = " " * indent
            new_lines.append(f"{spaces}# Полный патч выполнения изосурфейса через PyMCubes на CPU и возврат на GPU\n")
            new_lines.append(f"{spaces}device = level.device\n")
            new_lines.append(f"{spaces}volume_np = level.detach().cpu().numpy()\n")
            new_lines.append(f"{spaces}verts, faces = mcubes.marching_cubes(volume_np, 0.0)\n")
            new_lines.append(f"{spaces}v_pos = torch.from_numpy(verts.copy()).to(device, dtype=torch.float32)\n")
            new_lines.append(f"{spaces}t_pos_idx = torch.from_numpy(faces.copy()).to(device, dtype=torch.long)\n")
            skip_original_device_lines = True
            continue

        # Пропускаем оригинальные строки переноса на девайс, так как мы сделали это выше в патче
        if skip_original_device_lines and ("device = level.device" in line or "v_pos = v_pos.to(device)" in line or "t_pos_idx = t_pos_idx.to(device)" in line):
            continue

        new_lines.append(line)

    with open(isosurface_path, "w") as f:
        f.writelines(new_lines)
    print("[Патч 1] Успешно применен надежный построчный патч PyMCubes с поддержкой CUDA!")

# 5. Патчим tsr/system.py, чтобы игнорировать несовпадение ключей весов модели (strict=False)
system_path = "tsr/system.py"
if os.path.exists(system_path):
    with open(system_path, "r") as f:
        sys_code = f.read()
    # Заменяем строгую загрузку весов на нестрогую
    sys_code = sys_code.replace(
        "model.load_state_dict(ckpt)",
        "model.load_state_dict(ckpt, strict=False)"
    )
    with open(system_path, "w") as f:
        f.write(sys_code)
    print("[Патч 2] Успешно добавлена нестрогая загрузка весов!")

# 6. Запуск Gradio
!python app.py

/content
Cloning into 'TripoSR-hf'...
remote: Enumerating objects: 102, done.
remote: Counting objects: 100% (102/102), done.
remote: Compressing objects: 100% (70/70), done.
remote: Total 102 (delta 34), reused 95 (delta 30), pack-reused 0 (from 0)
Receiving objects: 100% (102/102), 444.55 KiB | 29.64 MiB/s, done.
Resolving deltas: 100% (34/34), done.
/content/TripoSR-hf
Found existing installation: onnxruntime 1.31.0
Uninstalling onnxruntime-1.31.0:
  Successfully uninstalled onnxruntime-1.31.0
[Патч 1] Успешно применен надежный построчный патч PyMCubes с поддержкой CUDA!
[Патч 2] Успешно добавлена нестрогая загрузка весов!
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://331f27657f90a220c1.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
